# YouTube Channel and Video Analysis

This notebook explores publicly available YouTube channel and video metadata with the YouTube Data API v3. It compares channel-level statistics, inspects the most-viewed videos for one selected channel, and summarizes upload timing.

## Research questions
- How do the selected channels differ in subscribers, lifetime views, and number of videos?
- Which videos in a selected channel have the most views?
- How are the selected channel's uploads distributed across months?

## Security and reproducibility
- Set `YOUTUBE_API_KEY` in your local environment. Never paste an API key into a notebook or commit it to Git.
- The notebook uses a small, curated set of channel IDs. API results change over time and are not included in the repository.
- Results are descriptive. They do not establish why a video was popular or that upload timing caused higher views.
- Use the YouTube API in accordance with its terms and quotas.

In [ ]:
import os
from pathlib import Path
import calendar

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from googleapiclient.discovery import build

sns.set_theme(style="whitegrid")
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

api_key = os.getenv("YOUTUBE_API_KEY")
if not api_key:
    raise RuntimeError(
        "Set YOUTUBE_API_KEY in your environment before running this notebook."
    )

youtube = build("youtube", "v3", developerKey=api_key)

## 1. Select channels

These are a small subset of the public channels that appeared in the original exploratory notebook. Edit the list for your own comparison.

In [ ]:
channel_ids = [
    # Add only public channel IDs you have a legitimate reason to analyze.
    "UCLx2uobmMEWyVpau81tT_yg",
    "UC-Q1yyL-PWpuaZ5GDRVhtCA",
    "UC7m6hFhBLb32DQqZTnlp-5Q",
    "UCh5wgwwsHnW4038EORUiuXw",
    "UC78cxCAcp7JfQPgKxYdyGrg",
    "UC4wL82JgBYScl9u263nC3cw",
    "UCD9kccBo3Choq3iy3bvvi7Q",
    "UCz9F9eEkt2KcLXZt--M6vjA",
    "UC64KwGynvj3hA7T0C9gC5cA",  # Thewizardliz in the original notebook
    "UCvrrZ9eseneke2UVe01ob2g",
    "UCqJODva8CJ_JTbi5ssC2L-g",
    "UCJS0K01yC8BmQtSuRdqqo3w",
    "UCSJ4gkVC6NrvII8umztf0Ow",
    "UC4XjuBdjIWGO7i6vck5ugug",
    "UCSOtPafGiITiWUKNzExwgwA",
    "UCuBuXy1NVoGXB3xC40kJ3_w",
    "UC3J-A2b-X7j2kgfXdjH5C2Q",
]

## 2. Collect channel statistics

The API accepts batches of up to 50 channel IDs per request. The helper below returns a tidy table for analysis.

In [ ]:
def get_channel_statistics(client, ids):
    records = []
    for start in range(0, len(ids), 50):
        response = (
            client.channels()
            .list(
                part="snippet,contentDetails,statistics",
                id=",".join(ids[start : start + 50]),
                maxResults=50,
            )
            .execute()
        )
        for item in response.get("items", []):
            stats = item.get("statistics", {})
            details = item.get("contentDetails", {})
            records.append(
                {
                    "channel_id": item["id"],
                    "channel_name": item["snippet"]["title"],
                    "subscribers": stats.get("subscriberCount"),
                    "views": stats.get("viewCount"),
                    "total_videos": stats.get("videoCount"),
                    "uploads_playlist_id": details.get("relatedPlaylists", {}).get("uploads"),
                }
            )
    return pd.DataFrame(records)


channel_data = get_channel_statistics(youtube, channel_ids)
for column in ["subscribers", "views", "total_videos"]:
    channel_data[column] = pd.to_numeric(channel_data[column], errors="coerce")
channel_data["views_per_video"] = (
    channel_data["views"] / channel_data["total_videos"].replace(0, float("nan"))
)
channel_data = channel_data.sort_values("subscribers", ascending=False, na_position="last")
channel_data.head()

In [ ]:
channel_data.to_csv(OUTPUT_DIR / "channel_statistics.csv", index=False)

## 3. Compare channels

The plots below show descriptive differences for the selected channel sample. For readability, the charts display the ten channels with the highest value for each metric.

In [ ]:
def plot_top_channels(data, metric, title):
    plot_data = data.nlargest(10, metric).sort_values(metric, ascending=True)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=plot_data, x=metric, y="channel_name")
    plt.title(title)
    plt.xlabel(metric.replace("_", " ").title())
    plt.ylabel("Channel")
    plt.tight_layout()
    plt.show()


plot_top_channels(channel_data, "subscribers", "Top selected channels by subscribers")
plot_top_channels(channel_data, "views", "Top selected channels by lifetime views")
plot_top_channels(channel_data, "total_videos", "Top selected channels by published video count")

## 4. Inspect videos for one channel

Choose a target by channel name. The lookup is case-insensitive, but the name must be present in the channel data returned by the API.

In [ ]:
TARGET_CHANNEL_NAME = "Thewizardliz"

matches = channel_data[
    channel_data["channel_name"].str.lower() == TARGET_CHANNEL_NAME.casefold()
]
if matches.empty:
    raise ValueError(
        f"{TARGET_CHANNEL_NAME!r} was not returned. Check the channel ID list and API response."
    )

target_channel = matches.iloc[0]
uploads_playlist_id = target_channel["uploads_playlist_id"]
target_channel[["channel_name", "channel_id", "uploads_playlist_id"]]

In [ ]:
def get_video_ids(client, playlist_id):
    video_ids = []
    page_token = None
    while True:
        response = (
            client.playlistItems()
            .list(
                part="contentDetails",
                playlistId=playlist_id,
                maxResults=50,
                pageToken=page_token,
            )
            .execute()
        )
        video_ids.extend(
            item["contentDetails"]["videoId"] for item in response.get("items", [])
        )
        page_token = response.get("nextPageToken")
        if not page_token:
            break
    return video_ids


video_ids = get_video_ids(youtube, uploads_playlist_id)
len(video_ids)

In [ ]:
def get_video_details(client, ids):
    records = []
    for start in range(0, len(ids), 50):
        response = (
            client.videos()
            .list(
                part="snippet,statistics",
                id=",".join(ids[start : start + 50]),
                maxResults=50,
            )
            .execute()
        )
        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            stats = item.get("statistics", {})
            records.append(
                {
                    "video_id": item["id"],
                    "title": snippet.get("title", ""),
                    "published_at": snippet.get("publishedAt"),
                    "views": stats.get("viewCount"),
                    "likes": stats.get("likeCount"),
                    "comments": stats.get("commentCount"),
                }
            )
    return pd.DataFrame(records)


video_data = get_video_details(youtube, video_ids)
video_data["published_at"] = pd.to_datetime(video_data["published_at"], errors="coerce", utc=True)
for column in ["views", "likes", "comments"]:
    video_data[column] = pd.to_numeric(video_data[column], errors="coerce")
video_data = video_data.sort_values("views", ascending=False, na_position="last")
video_data.head()

In [ ]:
video_data.to_csv(OUTPUT_DIR / "video_statistics.csv", index=False)

## 5. Most-viewed videos

In [ ]:
top_videos = video_data.dropna(subset=["views"]).nlargest(10, "views").sort_values("views")
plt.figure(figsize=(11, 7))
sns.barplot(data=top_videos, x="views", y="title")
plt.title(f"Most-viewed videos: {target_channel['channel_name']}")
plt.xlabel("Views at collection time")
plt.ylabel("Video title")
plt.tight_layout()
plt.show()

## 6. Upload timing by month

The count is based on the videos returned by the uploads playlist; it is not a measure of posting consistency adjusted for the channel's age or inactive periods.

In [ ]:
video_data["month"] = video_data["published_at"].dt.month
monthly_counts = (
    video_data.dropna(subset=["month"])
    .groupby("month")
    .size()
    .reindex(range(1, 13), fill_value=0)
    .rename("video_count")
    .reset_index()
)
monthly_counts["month_name"] = monthly_counts["month"].map(lambda m: calendar.month_abbr[int(m)])

plt.figure(figsize=(10, 5))
sns.barplot(data=monthly_counts, x="month_name", y="video_count", order=list(calendar.month_abbr)[1:])
plt.title(f"Uploads by calendar month: {target_channel['channel_name']}")
plt.xlabel("Month")
plt.ylabel("Videos in retrieved playlist")
plt.tight_layout()
plt.show()

## Limitations and next steps

- Metrics such as subscribers and views change after collection. Record a collection timestamp when doing comparisons.
- The selected channel sample is not random and should not be treated as representative of YouTube.
- Engagement and views are affected by many factors not measured here, including audience size, video age, topic, and platform recommendations.
- Some statistics may be omitted depending on API visibility and settings.
- Future work could normalize views by video age, use a documented sampling strategy, and compare channel-level patterns with uncertainty intervals.